# Retrouver la partition de piano d'un morceau : comparaison des approches

**Objectif.** Dans le générateur de vidéos TikTok (mode *Plateformes*), chaque rebond de la balle joue une note de piano.
Pour que la musique soit reconnaissable, il faut extraire du morceau **les bonnes notes** (la mélodie) et **les bons accords**,
au bon moment, même quand plusieurs instruments jouent en même temps.

Ce notebook retrace tous les essais, dans l'ordre : leur source, leur principe, leurs particularités, les résultats mesurés
et le verdict à l'écoute.

**Protocole commun (essais A à G).**
- Même extrait pour tous : la **première minute** de *Pirates des Caraïbes* (Hans Zimmer / Klaus Badelt, version orchestrale).
  Le fichier audio n'est pas versionné (droits d'auteur) : placer son propre MP3 dans `VideoResult/comparaison/pirates.mp3`.
- Toutes les versions sont rejouées avec **le même piano synthétisé** (`source/piano.py`) : seules les notes changent.
- Calculs sur **processeur seul** (16 cœurs, 8 Go alloués à Docker), sans carte graphique.
- Les rendus audio sont dans `VideoResult/comparaison/` (`A_actuel.mp3` … `G_sheetsage2.mp3`).
- Référence pour juger : le thème commence par **D D D E F F F G | E E D C C D** (ré mineur), sur la grille **Dm – B♭ – Am – Dm – B♭ – F – C – Dm**.

## Synthèse

| Essai | Méthode | Source | Notes de mélodie / min | Calcul (60 s de musique) | Verdict à l'écoute |
|---|---|---|---|---|---|
| 0a | Accords par chroma + tonalité, un accord par temps | librosa (CQT, chroma) + profils de Krumhansl | 1 accord / rebond | < 5 s | Juste mais « on ne reconnaît pas la musique » |
| 0b | Basic Pitch, toutes les notes | Spotify Basic Pitch | ~460 (tout) | ~15 s | Trop de notes, tous instruments mélangés |
| **A** | Basic Pitch + extraction de la mélodie | Basic Pitch + algorithme maison | ~150 | ~1 s* | Notes justes mais **beaucoup trop peu** |
| **B** | Demucs puis Basic Pitch par piste | Meta Demucs (htdemucs) + Basic Pitch | ~491 (avec accords) | ~59 s | **Bon timing**, mais **se trompe souvent de note** |
| **C** | Pop2Piano (reprise au piano générée) | Pop2Piano (Choi & Lee, ICASSP 2023) | 605 (tout) | ~23 s | **Invente trop** |
| **D** | Hybride : moments de B, hauteurs du mélange | B + transcription sensible du mélange | 227 | ~21 s + Demucs | Mieux, **pas parfait** |
| **E** | D + choix global de la ligne (Viterbi) | Viterbi + statistiques d'intervalles | 227 | ~21 s + Demucs | **Pas parfait**, certains enchaînements faux |
| **F** | YourMT3+ (transcription par instrument) | YourMT3+ (Chang et al., 2024) | 162 | ~132 s | **Très mauvais** |
| **G** | **Sheet Sage 2** (partition mélodie + accords) | m-a-p/SheetSage2 (2026), sur MERT-v2 | **245** (+ accords) | **~108 s** | ✅ **Parfait** → intégré à l'application |

\* transcription déjà en cache pour A ; ~15 s sinon.

## Préparation (optionnelle) : reproduire les essais

Chaque essai a tourné dans un conteneur Docker. Les recettes sont dans `experiments/transcription/docker/`,
les scripts dans `experiments/transcription/`. Les cellules ci-dessous servent à **réécouter** les résultats et à
**afficher les notes** ; les commandes Docker de chaque section permettent de tout recalculer.

In [ ]:
from pathlib import Path
import IPython.display as ipd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "transcription" else Path.cwd()
RESULTS = ROOT / "VideoResult" / "comparaison"

def listen(name):
    """Lecteur audio pour un rendu de la comparaison (s'il a été calculé sur cette machine)."""
    path = RESULTS / name
    if path.exists():
        return ipd.Audio(str(path))
    print(f"{path} absent : relancer l'essai correspondant (voir la commande de la section)")

RESULTS

## Essai 0a : un accord par temps, détecté par chroma

**Source.** Analyse du signal avec [librosa](https://librosa.org) : transformée à Q constant (CQT), chroma (énergie des
12 notes), estimation de la tonalité par les profils de **Krumhansl-Kessler**. Code : `source/piano.py` (`detect_chords`).

**Principe.** Les rebonds suivent le tempo (librosa `beat_track`). Entre deux rebonds, l'énergie des 12 notes est
comparée aux 24 accords majeurs / mineurs ; l'accord le plus proche est joué, avec sa basse et des renversements qui
bougent le moins possible.

**Particularités et améliorations successives.**
- Biais corrigé : l'analyse couvrait C2–C6 *inclus*, donc la note **C comptait sur 5 octaves** contre 4 pour les autres → des « C » et « Cm » parasites.
- Séparation harmonique / percussive (HPSS) avant l'analyse, chroma construit seulement sur les **pics** du spectre, bonus aux accords de la tonalité, hystérésis pour éviter les allers-retours.
- Tests synthétiques (batterie seule, ambiant sans rythme, tempo 90 → 150 BPM, 200 BPM) : tous corrects après corrections.

**Résultat.** Sur *Pirates* : grille **Dm – Am – B♭ – F – C – Gm**, juste.

**Verdict.** Harmonie juste, mais un seul accord par temps : **la mélodie n'y est pas**, on ne reconnaît pas le morceau.

## Essai 0b : Basic Pitch, toutes les notes de tous les instruments

**Source.** [Basic Pitch](https://github.com/spotify/basic-pitch) (Spotify, ICASSP 2022) : petit réseau de neurones de
transcription polyphonique, instrument-agnostique, modèle ONNX (pas besoin de TensorFlow).

**Principe.** Toutes les notes transcrites sont regroupées en rebonds (notes qui commencent ensemble = un accord,
écart minimum 0,15 s), et le piano rejoue toutes les notes avec leur durée et leur intensité.

**Particularités.** Rapide (~15 s pour 30 s de musique). Mélange tout : basse, accompagnement, cordes, cuivres…
Sur 30 s de *Pirates* : **228 notes → 107 rebonds**.

**Verdict.** Trop de notes, mélange de tous les instruments : il faut **choisir** les meilleures.

## Essai A : Basic Pitch + extraction de la mélodie principale

**Source.** Basic Pitch (ci-dessus) + algorithme maison `extract_melody` (`source/transcribe.py`).

**Principe.** Parmi les notes transcrites, à chaque instant on garde **la plus marquante** :
importance = intensité × √durée × poids de tessiture, avec une pénalité pour les grands sauts.
Puis chaque note est ramenée à l'octave la plus proche de la précédente (un thème doublé grave + aigu faisait sauter
la mélodie de registre). Accompagnement : l'accord de l'ensemble, rejoué doucement quand il change.

**Particularités.**
- Premier réglage trop strict (tessiture A3–E6 + sélectivité 0,35) : 472 → 96 notes / min. Élargi à C3–E6 et sélectivité 0,1 → **150 notes / min**.
- Le thème de *Pirates* est joué par les **cordes graves** : chercher la mélodie trop haut la faisait disparaître.

**Résultats (1 min).** 366 notes jouées (mélodie + accords doux), calcul ~1 s (transcription en cache).
Extrait 20–30 s : `F4 D4 A3 C4 C4 D4 D4 D4 A3 A3 A3 D4 G4 G4 A#4 D5 A4 F4 D4`

**Verdict.** « Joue juste mais **quasiment aucune note** » : Basic Pitch, sur un orchestre, rate beaucoup de notes.

In [ ]:
# Recalcul (A, B et C) : docker build -t tiktok-abtest -f docker/Dockerfile.abtest docker
# docker run --rm -v $PWD/../../source:/app/source -v $PWD/../../VideoResult/comparaison:/out -v $PWD:/t \
#            tiktok-abtest python /t/compare.py /out/pirates.mp3 /out 0 60
listen("A_actuel.mp3")

## Essai B : Demucs (séparation des instruments) puis Basic Pitch par piste

**Source.** [Demucs](https://github.com/facebookresearch/demucs) (Meta, modèle `htdemucs`) : séparation de sources en
4 pistes (voix, batterie, basse, autres). Approche classique « séparer puis transcrire » de la littérature sur
l'extraction de mélodie (voir par ex. les travaux de Durrieu et al. sur la séparation et transcription de la mélodie principale).

**Principe.** La mélodie est prise sur la piste **voix** si elle est présente, sinon sur **autres** (cordes, cuivres…).
Ici : énergie voix = 0,006, autres = 0,099, basse = 0,115 → mélodie prise sur « autres ». Accords : basse + autres.

**Particularités.** Les notes ne sont plus noyées sous les autres instruments, mais la séparation laisse des
**artefacts** (bavures, harmoniques) que Basic Pitch transcrit comme de fausses hauteurs.

**Résultats (1 min).** 491 notes jouées, calcul ~59 s (séparation comprise).
Extrait 20–30 s : `D4 F4 G4 E4 C4 C4 C4 D4 C4 D4 D4 C4 A3 A3 A3 C4 E4 C4 C4 D4 D4 D4 D4 G4 D4 G4 …`

**Verdict.** « Joue les notes **au bon moment** mais **se trompe souvent de note**. »

In [ ]:
listen("B_demucs.mp3")

## Essai C : Pop2Piano, reprise au piano générée directement

**Source.** [Pop2Piano: Pop Audio-based Piano Cover Generation](https://arxiv.org/abs/2211.00895) (Choi & Lee, ICASSP 2023),
modèle `sweetcocoa/pop2piano` dans [transformers](https://huggingface.co/docs/transformers/model_doc/pop2piano).
Transformer encodeur-décodeur (T5) entraîné sur des paires {morceau pop, reprise au piano jouée par un humain}.

**Principe.** Pas d'extraction de mélodie ni d'accords : le modèle **génère** une partition de piano (main droite +
main gauche) à partir de l'audio.

**Particularités.**
- Dépendance `essentia` compilée pour numpy 1.x : il a fallu forcer `numpy<2` dans le conteneur.
- Entraîné sur de la **pop** : sur un morceau orchestral, il arrange plus qu'il ne transcrit.
- Notes hors tonalité (G#2, D#4 en ré mineur) et nombreux arpèges.

**Résultats (1 min).** 605 notes, calcul ~23 s.
Extrait 20–30 s : `A#3 F4 G#2 G3 C4 A#3 F4 A2 G3 C4 E4 G4 …` (accords arpégés B♭, C…)

**Verdict.** « Pop2Piano **invente trop**. »

In [ ]:
# docker run ... tiktok-abtest sh -c "pip install 'numpy<2' && python /t/only_c.py /out/pirates.mp3 /out 0 60"
listen("C_pop2piano.mp3")

## Essai D : hybride, moments de Demucs + hauteurs du morceau complet

**Source.** Combinaison maison de B (timing) et de A (hauteurs), avec une transcription Basic Pitch **plus sensible**
du mélange (`onset_threshold=0.3`, `frame_threshold=0.2` au lieu de 0,5 / 0,3) : `d_hybrid.py`.

**Principe.** Chaque note de la mélodie de B donne **un moment**. À ce moment, on choisit la hauteur parmi les notes
du **morceau complet** : la plus audible, dans la tonalité, proche de la note précédente, avec un léger bonus si Demucs est d'accord.

**Particularités.**
- Premier essai raté : choisir « la note du mélange la plus proche de Demucs » **confirmait ses erreurs** (2 notes corrigées sur 227).
- Logique inversée : c'est le mélange qui décide → **27 notes corrigées sur 227** (12 %).

**Résultats (1 min).** 227 notes de mélodie (3,8 / s), calcul ~21 s + séparation Demucs.
Extrait 20–30 s : `D4 F4 G4 E4 C4 C4 C4 D4 C4 D4 D4 C4 A3 A3 A3 C4 C4 C4 C4 D4 F4 D4 D4 D4 G4 G4 G4 G4 …`

**Verdict.** « C'est **mieux** mais **pas parfait**. »

In [ ]:
# docker run ... tiktok-abtest python /t/d_hybrid.py /out      (réutilise la séparation de B)
listen("D_hybride.mp3")

## Essai E : choix global de la ligne mélodique (Viterbi)

**Source.** Algorithme de **Viterbi** (programmation dynamique, classique en transcription de mélodie depuis les
modèles de Markov cachés) ; probabilités d'intervalles inspirées des statistiques des mélodies occidentales
(beaucoup de notes répétées et de secondes, peu de grands sauts) : `e_viterbi.py`.

**Principe.** Au lieu de choisir chaque note en regardant seulement la précédente, on garde plusieurs candidates à
chaque moment et on choisit **la suite la plus cohérente sur toute la phrase** :
présence dans le morceau × note de l'accord en cours × note de la tonalité × probabilité de l'intervalle.

**Particularités.**
- 1er réglage : la régularité pesait trop → **70 % de notes répétées**, mélodie plate (D4 D4 D4 C4 C4 C4…).
- Rééquilibré (ce qu'on entend compte double) : 58 % répétées, 18 % de secondes, 0 % de sauts > quinte, 2 % hors tonalité.

**Résultats (1 min).** 227 notes, calcul ~21 s + Demucs.
Extrait 20–30 s : `F4 F4 D4 C4 C4 C4 C4 D4 C4 D4 A#3 C4 C4 C4 A3 A3 G3 C4 D4 D4 D4 …`

**Verdict.** « **Pas parfait**, certains enchaînements ne sont pas bons. » La limite vient des hauteurs candidates :
l'algorithme ne peut pas choisir une note que la transcription n'a pas vue.

In [ ]:
# docker run ... tiktok-abtest python /t/e_viterbi.py /out
listen("E_viterbi.mp3")

## Essai F : YourMT3+, transcription instrument par instrument

**Source.** [YourMT3+: Multi-instrument Music Transcription with Enhanced Transformer Architectures and Cross-dataset
Stem Augmentation](https://arxiv.org/abs/2407.04822) (Chang et al., 2024), descendant de MT3 (Google). Base des meilleurs
systèmes du [concours de transcription multi-instruments 2025](https://arxiv.org/abs/2603.27528) (F1 ≈ 0,83 sur Slakh2100).
Code et poids : [github.com/mimbres/YourMT3](https://github.com/mimbres/YourMT3), modèle « YPTF.MoE+Multi (noPS) ».

**Principe.** Le modèle sort une partition **par instrument** ; on prend l'instrument le plus présent dans le registre
médium-aigu (hors basses) et on en tire la mélodie.

**Particularités (mise en route difficile).**
- Code de 2024 : `transformers` épinglé en 4.44 (module retiré ensuite), `torchaudio.info/load` remplacés par soundfile.
- Checkpoint au format PyTorch < 2.6 : chargement complet (`weights_only=False`) autorisé dans le conteneur de test uniquement.
- **Manque de mémoire** (8 Go) avec 8 segments à la fois → traitement segment par segment.
- Instruments trouvés : String Ensemble (1 380 notes), Lead (193), basse (42), guitare (35), piano (30), saxophone (11).

**Résultats (1 min).** 162 notes de mélodie, calcul ~132 s.
Extrait 20–30 s : `F4 F4 G4 E4 E4 E4 C4 C4 D4 A#3 D4 A#3 E4 F4 A4 F4 G4 E4 E4 D4 C4 D4 …`

**Verdict.** « Le modèle F est **très mauvais**. » Entraîné surtout sur des jeux de données multipistes (Slakh, MusicNet…),
il se perd sur un enregistrement orchestral réel mixé, et choisir « l'instrument de la mélodie » reste fragile.

In [ ]:
# docker build -t tiktok-ymt3 -f docker/Dockerfile.yourmt3 docker
# (dépôt : git clone https://huggingface.co/spaces/mimbres/YourMT3-cpu ymt3 ; git lfs pull du checkpoint)
# docker run ... -e WANDB_MODE=disabled tiktok-ymt3 python /t/f_yourmt3.py /out
listen("F_yourmt3.mp3")

## Sheet Sage (version 1) : non testable

**Source.** [Melody transcription via generative pre-training](https://arxiv.org/abs/2212.01884) (Donahue, Thickstun,
Liang, ISMIR 2022), [github.com/chrisdonahue/sheetsage](https://github.com/chrisdonahue/sheetsage).

**Pourquoi il n'a pas été testé.**
1. L'image Docker officielle `chrisdonahue/sheetsage` est **cassée** sur Docker Hub (une couche n'existe plus : « not found »).
2. La recette pour la reconstruire part de `nvidia/cuda:10.1-cudnn7-devel-ubuntu18.04`, **retirée** de Docker Hub.
3. Sans le modèle Jukebox (carte graphique ≥ 12 Go), la qualité est dégradée ; licence non commerciale.

→ Remplacé par sa successeure, Sheet Sage 2.

## Essai G : Sheet Sage 2, partition « mélodie + accords » ✅

**Source.** [m-a-p/SheetSage2](https://huggingface.co/m-a-p/SheetSage2) (projet YuE2, 2026), rapport technique
*SheetSage2: Coherent Lead-Sheet Transcription with Synthetic Supervision*. Construit sur le modèle audio
[MERT-v2-FullSong](https://huggingface.co/m-a-p/MERT-v2-FullSong) avec des décodeurs structurés.
**Licence CC BY-NC 4.0 : usage non commercial uniquement.**

**Principe.** Le modèle est entraîné à produire une **partition simplifiée** (*lead sheet*), comme un musicien qui
relève un morceau : mélodie (vocale **ou** instrumentale), accords (y compris enrichis : Gm7, Asus4, B♭maj7),
temps et débuts de mesure, tonalité et structure (intro, couplet, interlude…). Sorties : ABC, MIDI séparés, fichiers .lab.

**Particularités (mise en route).**
- Versions exigées (PyTorch 2.8, transformers 4.45, numpy 1.24) incompatibles avec le reste → **image séparée**.
- Bug de `transformers` : les modules du modèle ne sont pas tous copiés dans son cache → copie manuelle des fichiers `.py`.
- Le modèle parent MERT-v2 est téléchargé au premier lancement (aucun compte Hugging Face nécessaire).

**Résultats (1 min).**
- Tonalité **ré mineur** ; mélodie **instrumentale** (aucune voix détectée, ce qui est juste) ; tempo 144 à la noire.
- Partition (ABC) : `D3D3DEF3F3FG | E3E3DCCD3…` = **D D D E F F F G | E E D C C D**, exactement le thème.
- Accords : Dm – B♭ – Am – Dm – B♭ – F – C – Dm – Gm7/B♭ – Gm – Dm – B♭ – Dm – A…
- **245 notes de mélodie** (4,1 / s) + 263 notes d'accords. Calcul du modèle : **108 s** (323 s au total la première fois, téléchargement et chargement compris).

**Verdict.** ✅ « Le G est **parfait**. » → intégré à l'application.

In [ ]:
# Essai isolé : docker build -t tiktok-ss2 -f docker/Dockerfile.sheetsage2 docker
# docker run ... tiktok-ss2 bash -c "cp /t/SheetSage2/*.py <cache modules> && python infer.py /out/extrait.wav --output /out/G_sheetsage2_out --dtype fp32 --device cpu"
# docker run ... tiktok-abtest python /t/g_render.py      (rejoue mélodie + accords avec notre piano)
listen("G_sheetsage2.mp3")

In [ ]:
# Partition produite par Sheet Sage 2 (format ABC) et grille d'accords
score = RESULTS / "G_sheetsage2_out" / "score.abc"
if score.exists():
    print(score.read_text()[:900])
    print("Accords :", " ".join(line.split()[2] for line in (RESULTS / "G_sheetsage2_out" / "chord.lab").read_text().splitlines()[:24]))

## Ce qu'on retient

1. **La qualité vient du modèle, pas du post-traitement.** Filtres de mélodie, hybrides et Viterbi (A, D, E) améliorent
   à la marge, mais ne peuvent pas inventer des notes que la transcription n'a pas vues, ni corriger des hauteurs fausses.
2. **Transcrire n'est pas relever une partition.** Basic Pitch, Demucs et YourMT3+ cherchent *toutes* les notes ;
   Pop2Piano *réinvente* un arrangement. Sheet Sage 2 est entraîné à faire ce qu'on veut vraiment : **la mélodie et les accords**
   qu'un musicien noterait.
3. **Coût.** Sheet Sage 2 est le plus lent sur processeur (~2 min par minute de musique) et non commercial ; d'où son
   intégration en **service séparé avec cache** : chaque morceau n'est transcrit qu'une fois.

## Intégration dans l'application

- `web/transcriber/` : service Flask + Sheet Sage 2 (image dédiée, modèle chargé une fois, résultats en cache par empreinte du fichier et extrait).
- `source/sheetsage_client.py` : appel depuis le rendu ; `source/platform_render.py` : moteur `engine="sheetsage2"` (par défaut) ou `"basic_pitch"`.
- Dans la vidéo : **un rebond par note de mélodie**, accords en accompagnement doux, fond qui pulse sur les **temps** détectés
  et petit zoom sur les **débuts de mesure**, couleur des plateformes = accord en cours.